# Example Model - Development Notebook

Data scientist development flow: EDA, training with StandardScaler, and prediction design.
Data loaded from local CSV/Parquet for initial development.

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")  # relative to notebooks/
train_path = DATA_DIR / "sample_train.csv"
df = pd.read_csv(train_path)
df.head()

## EDA

In [ ]:
df.describe()

In [ ]:
df["target"].value_counts()

## Training with StandardScaler

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib

FEATURE_COLUMNS = ["f0", "f1", "f2", "f3", "f4"]
X = df[FEATURE_COLUMNS]
y = df["target"]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_val_scaled)
accuracy = accuracy_score(y_val, y_pred)
print(f"Validation accuracy: {accuracy:.4f}")

## Save Artifacts (for validation)

In [ ]:
ARTIFACTS_DIR = Path("../artifacts/example/example_model/dev")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(model, ARTIFACTS_DIR / "model.pkl")
joblib.dump(scaler, ARTIFACTS_DIR / "scaler.pkl")
print("Saved model.pkl and scaler.pkl")

## Prediction Design

In [ ]:
inference_path = DATA_DIR / "sample_inference.csv"
inference_df = pd.read_csv(inference_path)
X_inf = inference_df[FEATURE_COLUMNS]
X_inf_scaled = scaler.transform(X_inf)
predictions = model.predict(X_inf_scaled)
result = inference_df.assign(prediction=predictions)
result